# 02 - Evaluate the Saved Model

MAE / RMSE are in production units (big crops dominate). `wape` is total error / total actual.
`median_ape` is the typical row's % error. Loads the `champion` model from the Unity Catalog Model Registry and scores the **test year** rows that already have a known
next-quarter value. It does not retrain anything.

In [0]:
import mlflow
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from configs.ml_config import (
    ALL_FEATURE_COLUMNS,
    BOOLEAN_FEATURES,
    FEATURE_TABLE,
    LABEL_COLUMN,
    MODEL_ALIAS,
    NUMERIC_FEATURES,
    REGISTERED_MODEL_NAME,
    TEST_YEAR,
)

In [0]:
mlflow.set_registry_uri("databricks-uc")

model_uri = f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}"
model = mlflow.sklearn.load_model(model_uri)
print(f"Loaded {model_uri}")

In [0]:
test_df = (
    spark.table(FEATURE_TABLE)
    .where(f"year = {TEST_YEAR} AND {LABEL_COLUMN} IS NOT NULL")
    .toPandas()
)
if test_df.empty:
    raise ValueError(f"No labeled rows for test year {TEST_YEAR}.")

for column in NUMERIC_FEATURES + BOOLEAN_FEATURES:
    test_df[column] = test_df[column].astype("float")

predicted = np.asarray(model.predict(test_df[ALL_FEATURE_COLUMNS]), dtype=float)
actual = test_df[LABEL_COLUMN].to_numpy(dtype=float)

absolute_error = np.abs(actual - predicted)
non_zero = actual != 0
metrics = {
    "rows": len(test_df),
    "mae": mean_absolute_error(actual, predicted),
    "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
    "wape": absolute_error.sum() / np.abs(actual).sum(),
    "median_ape": float(np.median(absolute_error[non_zero] / np.abs(actual[non_zero]))) if non_zero.any() else None,
    "r2": r2_score(actual, predicted),
}
print(f"Test year {TEST_YEAR}")
for metric_name, value in metrics.items():
    print(f"  {metric_name}: {value}")

## Error by crop group

In [0]:
scored_df = test_df[["province", "crop", "crop_group", "year", "quarter"]].copy()
scored_df["actual_production"] = actual
scored_df["predicted_production"] = predicted
scored_df["absolute_error"] = (scored_df["predicted_production"] - scored_df["actual_production"]).abs()

mae_by_crop_group = scored_df.groupby("crop_group")["absolute_error"].mean().rename("mae").reset_index()
print(mae_by_crop_group.to_string(index=False))
print("Next: 03_batch_inference")